# Padel Analytics — Custom Shot Classifier Training

This notebook shows how to train a simple **ML-based shot classifier**
as an upgrade over the rule-based engine in `shot_classifier.py`.

### What we build here
- Load shot events exported from the pipeline (`shots.csv`)
- Engineer features from wrist velocity + forearm angle
- Train a `RandomForestClassifier` (scikit-learn)
- Evaluate with cross-validation
- Save the trained model to `models/classification/shot_clf.pkl`

> **Note:** You need at least ~50 labelled shot events to train meaningfully.
> Run `main.py` on your video first to generate `data/outputs/shots.csv`.

---
## 0 · Setup

In [ ]:
import sys
import json
import pickle
from pathlib import Path

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path("../").resolve()
SRC  = ROOT / "src"
sys.path.insert(0, str(SRC))

from config import OUTPUTS_DIR, CLASSIFICATION_DIR

plt.style.use("dark_background")
plt.rcParams["figure.dpi"] = 110

print("Setup complete ✓")

---
## 1 · Load Shot Events

In [ ]:
CSV_PATH = ROOT / "data" / "outputs" / "shots.csv"

if not CSV_PATH.exists():
    print(f"ERROR: {CSV_PATH} not found.")
    print("Run main.py first to generate shot events.")
else:
    df = pd.read_csv(CSV_PATH)
    print(f"Loaded {len(df)} shot events from {CSV_PATH.name}")
    print(f"Columns: {list(df.columns)}")
    print(f"\nShot type distribution:")
    print(df["shot_type"].value_counts())
    display(df.head(10))

---
## 2 · Feature Engineering

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build ML features from raw shot event columns.

    Features
    --------
    wrist_velocity   : raw wrist speed (px/frame)
    forearm_angle    : elbow→wrist angle (degrees)
    abs_angle        : absolute value of forearm_angle
    angle_sign       : sign of forearm_angle (+1 or -1)
    vel_x_angle      : velocity × angle interaction
    confidence       : classifier's own confidence (meta-feature)
    """
    feat = pd.DataFrame()
    feat["wrist_velocity"] = df["wrist_velocity"]
    feat["forearm_angle"]  = df["forearm_angle"]
    feat["abs_angle"]      = df["forearm_angle"].abs()
    feat["angle_sign"]     = np.sign(df["forearm_angle"])
    feat["vel_x_angle"]    = df["wrist_velocity"] * df["forearm_angle"].abs()
    feat["confidence"]     = df["confidence"]
    return feat


if "df" in dir() and not df.empty:
    # Filter out 'unknown' shots for cleaner training
    df_clean = df[df["shot_type"] != "unknown"].copy()
    print(f"Rows after dropping 'unknown': {len(df_clean)}")

    X = engineer_features(df_clean)
    y = df_clean["shot_type"]

    print(f"\nFeature matrix shape: {X.shape}")
    print(f"Target classes      : {sorted(y.unique())}")
    display(X.describe().round(3))

---
## 3 · Correlation Heatmap

In [ ]:
if "X" in dir():
    fig, ax = plt.subplots(figsize=(7, 5))
    fig.patch.set_facecolor("#1a1a2e")
    ax.set_facecolor("#16213e")

    sns.heatmap(
        X.corr(),
        annot=True, fmt=".2f",
        cmap="coolwarm",
        ax=ax,
        linewidths=0.5,
        cbar_kws={"shrink": 0.8},
    )
    ax.set_title("Feature Correlation Matrix",
                 color="white", fontsize=12, pad=12)
    ax.tick_params(colors="white")
    plt.tight_layout()
    plt.savefig(
        str(ROOT / "data" / "outputs" / "chart_feature_correlation.png"),
        dpi=130, bbox_inches="tight",
        facecolor=fig.get_facecolor(),
    )
    plt.show()

---
## 4 · Train / Evaluate

In [ ]:
try:
    from sklearn.ensemble         import RandomForestClassifier, GradientBoostingClassifier
    from sklearn.model_selection  import StratifiedKFold, cross_val_score
    from sklearn.preprocessing    import LabelEncoder
    from sklearn.metrics          import classification_report, confusion_matrix
    from sklearn.pipeline         import Pipeline
    from sklearn.preprocessing    import StandardScaler
    _SK_AVAILABLE = True
    print("scikit-learn available ✓")
except ImportError:
    _SK_AVAILABLE = False
    print("scikit-learn not installed.")
    print("Run: pip install scikit-learn")

In [ ]:
if _SK_AVAILABLE and "X" in dir() and len(X) >= 10:

    # ── Label encode target ────────────────────────────────────────────────
    le = LabelEncoder()
    y_enc = le.fit_transform(y)
    print(f"Classes: {list(le.classes_)}")

    # ── Models to compare ─────────────────────────────────────────────────
    models = {
        "RandomForest":        RandomForestClassifier(
                                   n_estimators=200,
                                   max_depth=8,
                                   random_state=42,
                               ),
        "GradientBoosting":    GradientBoostingClassifier(
                                   n_estimators=100,
                                   learning_rate=0.1,
                                   random_state=42,
                               ),
    }

    cv  = StratifiedKFold(n_splits=min(5, len(X)), shuffle=True, random_state=42)
    results = {}

    for name, model in models.items():
        pipe   = Pipeline([("scaler", StandardScaler()), ("clf", model)])
        scores = cross_val_score(pipe, X, y_enc, cv=cv,
                                 scoring="accuracy", n_jobs=-1)
        results[name] = scores
        print(f"  {name:<22} acc={scores.mean():.3f} ± {scores.std():.3f}")

else:
    print("Skipping training — need scikit-learn + at least 10 labelled shots.")

In [ ]:
# ── Plot CV accuracy comparison ────────────────────────────────────────────────
if "results" in dir() and results:
    fig, ax = plt.subplots(figsize=(7, 4))
    fig.patch.set_facecolor("#1a1a2e")
    ax.set_facecolor("#16213e")

    names  = list(results.keys())
    means  = [results[n].mean() for n in names]
    stds   = [results[n].std()  for n in names]
    colours = ["#2196F3", "#FF9800"]

    bars = ax.bar(names, means, yerr=stds, capsize=6,
                  color=colours, edgecolor="#ffffff22")
    for bar, mean in zip(bars, means):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{mean:.3f}",
            ha="center", va="bottom",
            color="white", fontsize=10,
        )

    ax.set_ylim(0, 1.1)
    ax.set_ylabel("CV Accuracy", color="white")
    ax.set_title("Model Comparison (5-Fold CV)",
                 color="white", fontsize=12)
    ax.tick_params(colors="white")
    ax.spines[:].set_color("#ffffff33")
    plt.tight_layout()
    plt.show()

---
## 5 · Train Final Model & Save

In [ ]:
if _SK_AVAILABLE and "X" in dir() and len(X) >= 10:

    # ── Train on full dataset ─────────────────────────────────────────────
    best_model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    RandomForestClassifier(
                       n_estimators=200,
                       max_depth=8,
                       random_state=42,
                   )),
    ])
    best_model.fit(X, y_enc)

    # ── Classification report on full data ────────────────────────────────
    y_pred = best_model.predict(X)
    print("Classification report (full training data):")
    print(classification_report(
        y_enc, y_pred,
        target_names=le.classes_,
    ))

    # ── Confusion matrix ──────────────────────────────────────────────────
    cm  = confusion_matrix(y_enc, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    fig.patch.set_facecolor("#1a1a2e")
    sns.heatmap(
        cm, annot=True, fmt="d",
        xticklabels=le.classes_,
        yticklabels=le.classes_,
        cmap="Blues", ax=ax,
    )
    ax.set_xlabel("Predicted", color="white")
    ax.set_ylabel("Actual",    color="white")
    ax.set_title("Confusion Matrix", color="white", fontsize=12)
    ax.tick_params(colors="white")
    plt.tight_layout()
    plt.show()

    # ── Save model + label encoder ────────────────────────────────────────
    CLASSIFICATION_DIR.mkdir(parents=True, exist_ok=True)
    model_path = CLASSIFICATION_DIR / "shot_clf.pkl"
    le_path    = CLASSIFICATION_DIR / "label_encoder.pkl"

    with open(model_path, "wb") as f:
        pickle.dump(best_model, f)
    with open(le_path, "wb") as f:
        pickle.dump(le, f)

    print(f"\nModel saved      → {model_path}")
    print(f"Label encoder    → {le_path}")
    print("\nUpload both files to Google Drive and paste links in README.")

else:
    print("Skipping — need scikit-learn + labelled data.")

---
## 6 · Feature Importance

In [ ]:
if "best_model" in dir():
    rf   = best_model.named_steps["clf"]
    feat_names = list(X.columns)
    importances = rf.feature_importances_

    fig, ax = plt.subplots(figsize=(7, 4))
    fig.patch.set_facecolor("#1a1a2e")
    ax.set_facecolor("#16213e")

    sorted_idx = np.argsort(importances)
    ax.barh(
        [feat_names[i] for i in sorted_idx],
        importances[sorted_idx],
        color="#2196F3", edgecolor="#ffffff22",
    )
    ax.set_xlabel("Importance", color="white")
    ax.set_title("Feature Importance — RandomForest",
                 color="white", fontsize=12)
    ax.tick_params(colors="white")
    ax.spines[:].set_color("#ffffff33")
    plt.tight_layout()
    plt.savefig(
        str(ROOT / "data" / "outputs" / "chart_feature_importance.png"),
        dpi=130, bbox_inches="tight",
        facecolor=fig.get_facecolor(),
    )
    plt.show()